# Lab 4 — Classification : prédire une catégorie

### Objectifs du Lab

À la fin de ce Lab, l'étudiant sera capable de :

- **Entraîner** une régression logistique sur une tâche de classification binaire ;
- **Calculer et interpréter** accuracy, précision, rappel et F1-score ;
- **Ajuster le seuil de décision** d'un modèle et en observer l'effet sur précision et rappel ;
- **Observer** comment un déséquilibre des classes limite l’accuracy ;
- **Réutiliser** `validation croisée` pour régler l'hyperparamètre d'un modèle de classification.


### Lien avec le cours

Ce Lab correspond au chapitre 5 (*Modèles de classification*). Il réutilise directement les acquis des Labs précédents : évaluation et généralisation (Lab 1), pipeline de prétraitement (Lab 2). La matrice de confusion est définie pour la première fois dans ce Lab. Le fil conducteur est l'exemple de la **résiliation d'abonnement (*churn*)** introduit au chapitre 1 (§1.2) comme second exemple du triplet (T, P, E).


## 1. Construire un jeu de données de *churn*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
n = 500

anciennete_mois = rng.integers(1, 72, size=n)
appels_service_client = rng.poisson(1.5, size=n)
facture_mensuelle = rng.uniform(80, 600, size=n)
type_contrat = rng.choice(["Mensuel", "Annuel", "Deux ans"], size=n, p=[0.5, 0.3, 0.2])

# Probabilité de résiliation : augmente avec les appels au service client et
# les contrats mensuels, diminue avec l'ancienneté — une structure réaliste.
logit = (
    -2.0
    + 0.35 * appels_service_client
    - 0.03 * anciennete_mois
    + 0.004 * facture_mensuelle
    + pd.Series(type_contrat).map({"Mensuel": 1.2, "Annuel": -0.3, "Deux ans": -1.0}).to_numpy()
)
probabilite_churn = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, probabilite_churn)

clients = pd.DataFrame({
    "anciennete_mois": anciennete_mois,
    "appels_service_client": appels_service_client,
    "facture_mensuelle": facture_mensuelle.round(2),
    "type_contrat": type_contrat,
    "churn": churn,
})
print(clients["churn"].value_counts(normalize=True).round(3))


In [ ]:
from pathlib import Path

for dossier in ("data/raw", "data/interim", "data/processed", "outputs"):
    Path(dossier).mkdir(parents=True, exist_ok=True)


Comme souvent en pratique, les classes sont **déséquilibrées** : gardez ce chiffre en tête, il conditionne toute la suite (chapitre 2, §6, « le piège des classes déséquilibrées »).

## 2. Un pipeline de classification, de bout en bout

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = clients.drop(columns="churn")
y = clients["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y,
)

colonnes_numeriques = ["anciennete_mois", "appels_service_client", "facture_mensuelle"]
colonnes_categorielles = ["type_contrat"]

pretraitement = ColumnTransformer([
    ("num", StandardScaler(), colonnes_numeriques),
    ("cat", OneHotEncoder(drop="first"), colonnes_categorielles),
])

pipeline_logit = Pipeline([
    ("pretraitement", pretraitement),
    ("modele", LogisticRegression()),
])

pipeline_logit.fit(X_train, y_train)
print("Accuracy test :", round(pipeline_logit.score(X_test, y_test), 3))


## 3. Définir et lire la matrice de confusion

La matrice de confusion compare les classes réelles aux classes prédites. Les lignes représentent les classes réelles et les colonnes les classes prédites. La diagonale contient les bonnes prédictions ; les cases hors diagonale représentent les erreurs.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

y_pred = pipeline_logit.predict(X_test)
y_proba = pipeline_logit.predict_proba(X_test)[:, 1]

matrice = confusion_matrix(y_test, y_pred)
print(matrice)
print(classification_report(y_test, y_pred, target_names=["reste", "churn"], zero_division=0))

ConfusionMatrixDisplay(matrice, display_labels=["reste", "churn"]).plot(cmap="Blues")
plt.title("Matrice de confusion — churn")
plt.tight_layout()
plt.show()


La **précision** indique, parmi les clients signalés comme susceptibles de partir, combien partent réellement. Le **rappel** indique, parmi ceux qui partent réellement, combien le modèle a détectés. Le **F1-score** combine ces deux mesures.

## 4. Le seuil de décision n'est pas obligatoirement 0,5

Par défaut, `.predict()` classe en « churn » toute observation dont la probabilité prédite dépasse 0,5. Mais rien n'impose ce seuil : selon le coût métier d'un faux négatif (un client qui part sans qu'on ait rien tenté) par rapport à un faux positif (une offre de rétention envoyée à tort), on peut vouloir un seuil différent.

In [ ]:
from sklearn.metrics import precision_score, recall_score

for seuil in [0.3, 0.5, 0.7]:
    y_pred_seuil = (y_proba >= seuil).astype(int)
    precision = precision_score(y_test, y_pred_seuil)
    rappel = recall_score(y_test, y_pred_seuil)
    print(f"Seuil = {seuil} → précision = {precision:.3f}, rappel = {rappel:.3f}")


## 5. Des classes déséquilibrées

Quand une classe est rare, l’accuracy peut donner une impression trop favorable. Il faut aussi lire la précision, le rappel et le F1-score, puis comparer le modèle à la baseline présentée dans la section suivante.

`` donne automatiquement plus de poids à la classe minoritaire pendant l'entraînement, sans avoir à dupliquer ou supprimer des observations. Comparez le rappel de la classe « churn » avec et sans cette option : c'est généralement là que l'écart est le plus visible.

## 6. Comparer à une baseline

Comparez le modèle à une règle simple qui prédit toujours la classe majoritaire. N’utilisez pas le jeu de test pour choisir le seuil ou modifier le modèle.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print("Accuracy baseline :", round(baseline.score(X_test, y_test), 3))
print("Accuracy modèle   :", round(pipeline_logit.score(X_test, y_test), 3))


## 7. Synthèse

### Auto-évaluation

- [ ] Je sais lire les lignes et les colonnes d’une matrice de confusion.
- [ ] Je sais calculer et interpréter accuracy, précision, rappel et F1-score.
- [ ] Je sais faire varier le seuil de décision et expliquer son effet sur précision et rappel.
- [ ] Je sais comparer un modèle à une baseline.

## 8. Lab Sheet — Projet guidé

### Jeu de données du projet : le Titanic, à nouveau

Reprenez le jeu de données du Titanic préparé au Lab 2 (§9). Cette fois, l'objectif n'est plus seulement de le nettoyer, mais de réellement **prédire** la survie (`survived`) d'un passager.

In [ ]:
titanic = pd.read_csv(
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
)


### Partie A — Reprendre le pipeline du Lab 2

1. Réappliquez (ou reconstruisez) le pipeline de prétraitement du Lab 2 sur les variables `pclass`, `sex`, `age`, `fare`, `embarked`, `sibsp`, `parch`, avec `survived` comme cible. Séparez Train (75 %) / Test (25 %), stratifiés.
2. Le jeu est-il déséquilibré ? Donnez la proportion de survivants.

### Partie B — Premier modèle et métriques

3. Entraînez une `LogisticRegression` (pipeline complet). Calculez accuracy, précision, rappel et F1-score sur le test.
4. Affichez la matrice de confusion et décrivez les erreurs les plus fréquentes.
5. Une `DummyClassifier` (baseline) obtiendrait quelle accuracy sur ce jeu ? Votre modèle fait-il mieux ?

### Partie C — Seuil de décision

6. Testez trois seuils de décision, par exemple 0,3, 0,5 et 0,7. Comparez la précision et le rappel.
7. Si l’objectif prioritaire est de ne manquer aucun client susceptible de partir, quel seuil choisiriez-vous ? Quel compromis cela entraîne-t-il ?

### Partie D — Conclusion

8. Comparez les résultats du modèle à ceux de la baseline. L’amélioration vous paraît-elle suffisante ? Justifiez avec les métriques calculées.
9. Quelle erreur serait la plus coûteuse dans ce contexte : signaler à tort un client ou ne pas détecter un client qui part ? Reliez votre réponse à la précision et au rappel.

### Livrables

- un notebook ou script contenant le code des parties A à D ;
- une matrice de confusion dans `outputs/` ;
- un court paragraphe de conclusion, comparant les modèles testés et justifiant votre choix final.

### Checklist finale

- [ ] Le déséquilibre des classes est observé avant l’évaluation.
- [ ] Au moins trois métriques de classification (au-delà de l'accuracy) sont calculées.
- [ ] L'effet du seuil de décision est illustré avec au moins deux valeurs différentes.
- [ ] Le test est réservé à l’évaluation finale.
- [ ] Le jeu de test n'est utilisé qu'une seule fois, pour l'évaluation finale.